# 10 Ablation v1

This stage runs a focused Tier-1 ablation suite for the RL overlay project. It is intentionally small: the goal is to answer the highest-value research questions before launching any larger ablation campaign.

The four questions are:
- Is RL mainly dynamic control or fixed-parameter discovery?
- Does most RL value come from controlling lambda or tau?
- Does path-dependent state matter?
- Are conclusions robust to transaction-cost assumptions?

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
PYTHON = sys.executable

TEST_START = "2020-01"
TEST_END = "2024-11"
COST_BPS = 10.0
COST_GRID = "0,10,20,50"
FIXED_PARAM_SOURCE = "validation_mean"

OUTDIR = PROJECT_ROOT / "data/ablation_v1"
OUTDIR.mkdir(parents=True, exist_ok=True)

SCRIPT = PROJECT_ROOT / "scripts/run_tier1_ablation_v1.py"
PRED_FILE = PROJECT_ROOT / "data/prediction/fm_oos_predictions.parquet"
RISK_DIR = PROJECT_ROOT / "data/risk/risk_cov_npz"
RISK_META_FILE = PROJECT_ROOT / "data/risk/risk_monthly_metadata.csv"
RETURNS_FILE = PROJECT_ROOT / "data/panel/monthly_stock_panel_basic_full.parquet"
RL_BACKTEST = PROJECT_ROOT / "data/rl_overlay_sac/test_backtest.csv"
RL_WEIGHTS = PROJECT_ROOT / "data/rl_overlay_sac/test_weights.parquet"
RL_ACTIONS = PROJECT_ROOT / "data/rl_overlay_sac/test_action_history.csv"
TRAIN_ACTIONS = PROJECT_ROOT / "data/rl_overlay_sac/train_action_history.csv"
VALIDATION_ACTIONS = PROJECT_ROOT / "data/rl_overlay_sac/validation_action_history.csv"
FAIR_FIXED_BACKTEST = PROJECT_ROOT / "data/diagnostics/static_fixed_param_fair/static_fixed_fair_backtest.csv"
FAIR_FIXED_WEIGHTS = PROJECT_ROOT / "data/diagnostics/static_fixed_param_fair/static_fixed_fair_weights.parquet"
FAIR_FIXED_SUMMARY = PROJECT_ROOT / "data/diagnostics/static_fixed_param_fair/static_fixed_fair_summary.csv"

COMMON_ARGS = [
    "--project-root", str(PROJECT_ROOT),
    "--pred-file", str(PRED_FILE),
    "--risk-dir", str(RISK_DIR),
    "--risk-meta-file", str(RISK_META_FILE),
    "--returns-file", str(RETURNS_FILE),
    "--rl-backtest-file", str(RL_BACKTEST),
    "--rl-weights-file", str(RL_WEIGHTS),
    "--rl-action-history-file", str(RL_ACTIONS),
    "--train-action-history-file", str(TRAIN_ACTIONS),
    "--validation-action-history-file", str(VALIDATION_ACTIONS),
    "--fixed-param-source", FIXED_PARAM_SOURCE,
    "--fair-fixed-backtest-file", str(FAIR_FIXED_BACKTEST),
    "--fair-fixed-weights-file", str(FAIR_FIXED_WEIGHTS),
    "--fair-fixed-summary-file", str(FAIR_FIXED_SUMMARY),
    "--outdir", str(OUTDIR),
    "--cost-bps", str(COST_BPS),
    "--test-start", TEST_START,
    "--test-end", TEST_END,
]


In [ ]:
required_artifacts = {
    "ablation_script": SCRIPT,
    "predictions": PRED_FILE,
    "risk_dir": RISK_DIR,
    "risk_metadata": RISK_META_FILE,
    "returns": RETURNS_FILE,
    "rl_backtest": RL_BACKTEST,
    "rl_weights": RL_WEIGHTS,
    "rl_actions": RL_ACTIONS,
    "train_actions_for_fair_fixed": TRAIN_ACTIONS,
    "validation_actions_for_fair_fixed": VALIDATION_ACTIONS,
    "fair_fixed_backtest_if_precomputed": FAIR_FIXED_BACKTEST,
}

status = pd.DataFrame(
    [
        {
            "artifact": name,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
            "required_now": name not in {
                "fair_fixed_backtest_if_precomputed",
                "train_actions_for_fair_fixed",
                "validation_actions_for_fair_fixed",
            },
        }
        for name, path in required_artifacts.items()
    ]
)
display(status)

missing_required = status.loc[status["required_now"] & ~status["exists"], "path"].tolist()
if missing_required:
    raise FileNotFoundError(f"Missing required artifacts for Tier-1 ablations: {missing_required}")

fixed_param_inputs_available = FAIR_FIXED_BACKTEST.exists() or (
    (FIXED_PARAM_SOURCE == "train_mean" and TRAIN_ACTIONS.exists())
    or (FIXED_PARAM_SOURCE == "validation_mean" and VALIDATION_ACTIONS.exists())
    or FIXED_PARAM_SOURCE == "manual"
)
if not fixed_param_inputs_available:
    print(
        "Fair fixed/control ablations will write missing_artifact rows until the "
        "train/validation fixed-parameter source or precomputed fair benchmark exists."
    )


## A. Fair Fixed-Parameter Comparison

This compares the held-out RL overlay against a fair fixed-parameter benchmark. The fixed lambda/tau pair must be selected from train or validation action history only, then evaluated on the test window. If the fixed-parameter source is not available yet, the script records an explicit `missing_artifact` row rather than substituting a test-informed parameter.

In [ ]:
subprocess.run(
    [
        PYTHON,
        str(SCRIPT),
        "--ablation-type", "fair_fixed_param",
        *COMMON_ARGS,
    ],
    check=True,
)

fair_table = pd.read_csv(OUTDIR / "tier1_ablation_master_comparison.csv")
display(
    fair_table.loc[fair_table["ablation_type"] == "fair_fixed_param", [
        "run_name", "role", "status", "annualized_return", "sharpe_ratio",
        "max_drawdown", "cumulative_return", "mean_monthly_turnover",
        "mean_lambda", "mean_tau",
    ]]
)


## B. Control-Dimension Ablation

These runs keep the SAC test action path but freeze one control dimension at its ex-ante fixed value. Lambda-only keeps SAC's lambda path and fixes tau. Tau-only keeps SAC's tau path and fixes lambda.

In [ ]:
for ablation_type in ["lambda_only", "tau_only"]:
    subprocess.run(
        [
            PYTHON,
            str(SCRIPT),
            "--ablation-type", ablation_type,
            *COMMON_ARGS,
        ],
        check=True,
    )

control_table = pd.read_csv(OUTDIR / "tier1_ablation_master_comparison.csv")
display(
    control_table.loc[control_table["ablation_type"].isin(["fair_fixed_param", "lambda_only", "tau_only"]), [
        "ablation_type", "run_name", "role", "status", "annualized_return",
        "sharpe_ratio", "cumulative_return", "mean_monthly_turnover",
        "mean_lambda", "std_lambda", "mean_tau", "std_tau",
    ]]
)


## C. State Ablation

The full summary-state baseline is registered from the current RL overlay test artifact. Reduced-state variants can be added as soon as their trained artifacts exist under the expected paths or are passed explicitly to the script.

In [ ]:
state_variants = [
    "full_summary_state",
    "no_prev_return_turnover",
    "minimal_state",
]

for variant in state_variants:
    subprocess.run(
        [
            PYTHON,
            str(SCRIPT),
            "--ablation-type", "state_ablation",
            "--state-variant", variant,
            "--allow-missing-state-variant", "true",
            *COMMON_ARGS,
        ],
        check=True,
    )

state_table = pd.read_csv(OUTDIR / "tier1_ablation_master_comparison.csv")
display(
    state_table.loc[state_table["ablation_type"] == "state_ablation", [
        "run_name", "role", "status", "annualized_return", "sharpe_ratio",
        "cumulative_return", "mean_monthly_turnover", "mean_lambda", "mean_tau",
    ]]
)


## D. Cost Robustness Ablation

This revalues the same SAC action path under several transaction-cost assumptions. The optimizer decisions use the same lambda/tau path; the accounting changes through `cost_bps`.

In [ ]:
subprocess.run(
    [
        PYTHON,
        str(SCRIPT),
        "--ablation-type", "cost_robustness",
        "--cost-grid", COST_GRID,
        *COMMON_ARGS,
    ],
    check=True,
)

cost_table = pd.read_csv(OUTDIR / "tier1_ablation_master_comparison.csv")
display(
    cost_table.loc[cost_table["ablation_type"] == "cost_robustness", [
        "run_name", "cost_bps", "status", "annualized_return", "sharpe_ratio",
        "cumulative_return", "mean_monthly_turnover", "mean_cost",
    ]].sort_values("cost_bps")
)


## Master Tier-1 Comparison

The master table aggregates all Tier-1 runs created so far. Rows with `missing_artifact` are intentional placeholders for state variants that have not been trained yet.

In [ ]:
master = pd.read_csv(OUTDIR / "tier1_ablation_master_comparison.csv")
cols = [
    "ablation_type", "run_name", "role", "status", "cost_bps", "n_months",
    "annualized_return", "sharpe_ratio", "max_drawdown", "cumulative_return",
    "mean_monthly_turnover", "mean_cost", "mean_lambda", "std_lambda", "mean_tau", "std_tau",
]
display(master[cols].sort_values(["ablation_type", "run_name"]))


In [ ]:
plots = [
    OUTDIR / "tier1_cumulative_return_comparison.png",
    OUTDIR / "tier1_turnover_comparison.png",
    OUTDIR / "tier1_lambda_tau_comparison.png",
]
for plot_path in plots:
    if plot_path.exists():
        print(plot_path.relative_to(PROJECT_ROOT))
        display(Image(filename=str(plot_path)))


## Interpretation Notes

- Dynamic control vs fixed discovery: compare `rl_overlay_sac` with `static_fixed_param_fair`. If they are close, the apparent RL value may be mostly fixed-parameter discovery.
- Lambda vs tau control: compare `lambda_only_rl` and `tau_only_rl` with the full RL baseline. The closer run identifies the control dimension carrying more of the realized value.
- State dependence: compare reduced-state variants with `full_summary_state` once their trained artifacts exist. Placeholder rows mark variants that still need training artifacts.
- Cost robustness: read performance monotonically across 0, 10, 20, and 50 bps. A steep slope indicates the strategy is mainly turnover/cost constrained.
- The next deeper ablation should only expand dimensions that materially change the Tier-1 conclusions.